# 실습 2. 스트리밍 오류·연결 중단 대응 및 결과 점검

## 실습 목표

이번 실습은 기존에 만든 `/chat/stream/safe`를 출발점으로 삼아, 스트리밍 응답이 일반 응답과 다르게 깨지는 **네 가지 케이스(API 호출 실패, 빈 응답, 타임아웃, 클라이언트 중단)**를 직접 재현하고 서버에서 어떻게 대응할지 코드로 확인하는 단계입니다. 이번 실습이 끝나면 스트리밍 응답을 운영 환경에서 안전하게 다루는 기본기를 갖춥니다.

네 가지 케이스를 차례로 다뤄봅니다.

- API 호출 실패 / 빈 응답 / 타임아웃 / 클라이언트 중단을 각각 재현
- `try/except` + `finally` + 종료 마커(`[DONE]` / `[EMPTY]` / `[ERROR]`)로 본문 안에서 오류를 신호로 전달
- 클라이언트 중단(`CancelledError`)을 서버 generator에서 감지하고 `finally`로 정리 로그 남기기
- `status 200`만 보지 않고 본문 길이 · 종료 마커 · 서버 로그까지 함께 확인하는 검증 습관

## 1. 클라이언트 준비

이전 실습에서 쓰던 방식 그대로, MLAPI 접속 정보를 `.env`에서 읽습니다. `.env`에 값이 없으면 입력 프롬프트로 받아 `.env` 파일에 저장하고, server.py는 이 `.env`에서 키를 읽습니다. 키가 코드나 server.py에 직접 박히지 않습니다.

In [6]:
import os
from getpass import getpass
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

base_url   = os.getenv("MLAPI_BASE_URL", "https://mlapi.run/40cc17ae-a89b-4f12-a7d6-13293180fc87/v1")
api_key    = os.getenv("MLAPI_API_KEY")
model_name = os.getenv("MLAPI_MODEL", "openai/gpt-4o-mini")

if not api_key or api_key.startswith("여기에"):
    api_key = getpass("MLAPI API Key를 입력하세요: ")

# server.py가 별도 프로세스에서 읽을 수 있도록 .env 파일에 저장
Path(".env").write_text(
    f"MLAPI_BASE_URL={base_url}\n"
    f"MLAPI_API_KEY={api_key}\n"
    f"MLAPI_MODEL={model_name}\n",
    encoding="utf-8",
)
Path(".gitignore").write_text(".env\n__pycache__/\n", encoding="utf-8")

PORT = 8001                  # 서버 포트. 충돌나면 8001 등으로 변경
SERVER_BASE = f"http://localhost:{PORT}"

print("모델명:", model_name, "/ PORT:", PORT)
print("server base:", SERVER_BASE)
print(".env 저장 완료 — server.py가 이 파일에서 키를 읽습니다")

모델명: openai/gpt-5.4 / PORT: 8001
server base: http://localhost:8001
.env 저장 완료 — server.py가 이 파일에서 키를 읽습니다


## 2. server.py 자동 생성

이번 server.py 의 핵심 변화는 /chat/stream/safe 를 async 로 통일한 것입니다. sync 함수 + sync generator 로 두면 uvicorn 이 별도 스레드풀에서 돌리는데, 그 스레드가 OpenAI 응답을 블로킹으로 기다리고 있을 때 클라이언트가 끊어도 finally 가 곧장 실행되지 않습니다. async 로 바꾸면 클라이언트 끊김이 CancelledError 로 즉시 들어와 finally 가 바로 호출됩니다.

구성:

- ChatRequest Pydantic 모델 (실습 1-4 와 동일)
- /health
- /chat/stream/safe : async + AsyncOpenAI. 정상 종료 시 [DONE], 예외 시 [ERROR] ... 한 줄, finally 로그 한 줄
- /chat/stream/break : 빈 응답만 흘리는 더미 (TODO 2 에서 [EMPTY] 마커 추가)

이 셀은 파일을 쓰기만 합니다. 서버는 다음 단계에서 터미널로 직접 띄웁니다.

In [3]:
from pathlib import Path

# ── server.py 조각 ── head=공통 / safe·break_=TODO 1·2 에서 교체 / robust=TODO 3 에서 정의
head = '''import os
import asyncio
from typing import Optional
from dotenv import load_dotenv
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field
from openai import AsyncOpenAI

load_dotenv()
BASE_URL = os.getenv("MLAPI_BASE_URL")
API_KEY  = os.getenv("MLAPI_API_KEY")
MODEL    = os.getenv("MLAPI_MODEL", "openai/gpt-4o-mini")

if not BASE_URL:
    raise RuntimeError("MLAPI_BASE_URL이 설정되어 있지 않습니다. .env 파일을 확인하세요.")
if not API_KEY:
    raise RuntimeError("MLAPI_API_KEY가 설정되어 있지 않습니다. .env 파일을 확인하세요.")

client = AsyncOpenAI(api_key=API_KEY, base_url=BASE_URL)

app = FastAPI()


class ChatRequest(BaseModel):
    message: str = Field(..., min_length=1)
    model: Optional[str] = Field(None)
    temperature: Optional[float] = Field(None, ge=0.0, le=2.0)


@app.get("/health")
def health():
    return {"ok": True}
'''

safe = '''

@app.post("/chat/stream/safe")
async def chat_stream_safe(req: ChatRequest):
    async def gen():
        try:
            stream = await client.chat.completions.create(
                model=req.model or MODEL,
                messages=[{"role": "user", "content": req.message}],
                temperature=req.temperature if req.temperature is not None else 1.0,
                stream=True,
            )
            async for chunk in stream:
                if not chunk.choices:
                    continue
                piece = getattr(chunk.choices[0].delta, "content", None)
                if not piece:
                    continue
                yield piece
            yield "\\n[DONE]"
        except asyncio.CancelledError:
            print("[safe] cancelled by client")
            raise
        except Exception as e:
            yield f"\\n[ERROR] {type(e).__name__}: {e}"
        finally:
            print("[safe] generator finished or cancelled")
    return StreamingResponse(gen(), media_type="text/plain")
'''

break_ = '''

@app.post("/chat/stream/break")
async def chat_stream_break(req: ChatRequest):
    """빈 응답만 흘리는 더미 — TODO 2 에서 [EMPTY] 감지를 더한다."""
    async def gen():
        try:
            for _ in range(3):
                piece = ""  # 실제로는 LLM chunk 가 전부 빈 문자열인 상황
                if piece:
                    yield piece
        finally:
            print("[break] generator finished")
    return StreamingResponse(gen(), media_type="text/plain")
'''

robust = ""   # TODO 3 에서 채움

Path("server.py").write_text(head + safe + break_ + robust, encoding="utf-8")
print("server.py 작성 완료 (head + safe + break)")

server.py 작성 완료 (head + safe + break)


## 3. 서버 띄우기

터미널에서 아래 셀이 출력하는 명령어를 그대로 붙여넣어 실행하세요.

- --reload 덕분에 server.py 저장 시 자동 재시작
- 중지: 터미널에서 Ctrl + C
- 포트 충돌 시 2 단계 PORT 만 바꾸고 2~4 단계 셀 재실행
- 이번 실습은 서버 터미널 로그를 자주 봐야 합니다. 터미널 창을 노트북과 같이 보이게 배치해 두세요.

In [4]:
print(f"uvicorn server:app --reload --port {PORT}")
print()
print("서버 주소:", SERVER_BASE)

uvicorn server:app --reload --port 8000

서버 주소: http://localhost:8000


In [7]:
import httpx

r = httpx.get(f"{SERVER_BASE}/health", timeout=5.0)
print(r.status_code, r.json())

200 {'ok': True}


## 4. 오류 케이스 1 - API 호출 실패

가장 흔한 오류: 모델명 오타, 잘못된 키, 베이스 URL 미스매치 같은 호출 자체의 실패입니다.

/chat/stream/safe 의 generator 는 OpenAI 호출을 try 로 감싸 두었기 때문에, 호출 단계에서 예외가 나면 본문 끝에 [ERROR] ... 한 줄을 흘리고 정상 종료합니다. 서버 프로세스는 죽지 않아야 합니다.

**해볼 것**
- 노트북에서 일부러 잘못된 model 을 함께 보냄
- 응답 본문 마지막에 [ERROR] ... 한 줄이 보이는지 확인
- 서버 터미널에 [safe] generator finished or cancelled 가 찍히는지 확인 (finally 가 호출됨)

In [8]:
import httpx

with httpx.Client(timeout=60.0) as c:
    with c.stream(
        "POST",
        f"{SERVER_BASE}/chat/stream/safe",
        json={"message": "테스트", "model": "this-model-does-not-exist"},
    ) as r:
        print("status:", r.status_code)
        for piece in r.iter_text():
            print(piece, end="", flush=True)
print("\n--- end ---")

status: 200

[ERROR] BadRequestError: Error code: 400 - {'detail': "Model 'this-model-does-not-exist' is not allowed. Allowed models: ['openai/gpt-5.4']"}
--- end ---


## 5. 오류 케이스 2 - 빈 응답

호출 자체는 실패하지 않았는데 본문이 한 줄도 안 흐르는 경우가 있습니다. 모델이 빈 chunk 만 보내면 가드에 걸려 yield 가 한 번도 일어나지 않고, 그대로 generator 가 끝나 버립니다.

/chat/stream/break 는 정확히 그 상황을 시뮬레이션한 엔드포인트입니다. 아래 셀을 실행해 보면 알 수 있듯이, 응답은 status 200 으로 멀쩡하게 끝나고 본문 길이만 0 입니다. 클라이언트 입장에서는 에러가 난 것도 아니고, 그렇다고 결과가 있는 것도 아닙니다. "성공인데 결과가 없는 것" 과 "실패한 것" 을 한 줄 본문 검사만으로는 구분할 수 없습니다.

표준 대응은: generator 안에서 한 번도 yield 한 적이 없으면 [EMPTY] 같은 sentinel 을 마지막에 한 줄 흘려서, 받는 쪽이 "빈 응답" 임을 명시적으로 알 수 있게 하는 것입니다. 실제 적용은 TODO 2 에서 학생이 직접 합니다.

In [9]:
import httpx

with httpx.Client(timeout=30.0) as c:
    with c.stream(
        "POST",
        f"{SERVER_BASE}/chat/stream/break",
        json={"message": "비어있을 예정"},
    ) as r:
        print("status:", r.status_code)
        body = ""
        for piece in r.iter_text():
            body += piece
            print(piece, end="", flush=True)
print("\n--- end ---")
print("본문 길이:", len(body))
print("→ status 는 200, 본문 길이는 0. 에러 없이 '빈 응답' 이 정상 응답과 구분되지 않습니다. TODO 2 에서 [EMPTY] 마커로 구분해 줍니다.")

status: 200

--- end ---
본문 길이: 0
→ status 는 200, 본문 길이는 0. 에러 없이 '빈 응답' 이 정상 응답과 구분되지 않습니다. TODO 2 에서 [EMPTY] 마커로 구분해 줍니다.


## 6. 오류 케이스 3 - 타임아웃

타임아웃은 두 군데서 잡을 수 있습니다.

- 서버 쪽: client.chat.completions.create(..., timeout=...) 로 OpenAI 호출 자체에 제한
- 클라이언트 쪽: httpx.Client(timeout=...) 로 노트북에서의 대기 제한

둘 중 한쪽만 걸어도 무한 대기는 막을 수 있지만, 둘 다 거는 것이 안전합니다. 한쪽이 멈춰도 다른 쪽이 끊어 주니까.

여기서는 노트북 클라이언트 쪽 타임아웃부터 확인합니다. 일부러 아주 짧은 타임아웃을 걸어 보면, httpx.ReadTimeout 이 클라이언트에서 발생하는 것을 볼 수 있습니다. 서버 쪽 timeout 적용은 TODO 1 에서 직접 합니다.

In [10]:
import httpx

try:
    with httpx.Client(timeout=0.5) as c:  # 일부러 너무 짧게
        with c.stream(
            "POST",
            f"{SERVER_BASE}/chat/stream/safe",
            json={"message": "한국의 역사를 가능한 한 길게 설명해줘"},
        ) as r:
            for piece in r.iter_text():
                print(piece, end="", flush=True)
    print("\n--- end ---")
except httpx.TimeoutException as e:
    print(f"\n[client timeout] {type(e).__name__}: {e}")


[client timeout] ReadTimeout: timed out


클라이언트가 타임아웃으로 끊으면 서버 쪽 generator 도 깨집니다. 다음 단계에서 그 모습을 직접 봅니다.

## 7. 오류 케이스 4 - 클라이언트 중단

노트북에서 응답을 도중에 끊으면(루프에서 break) 서버 쪽 async generator 가 CancelledError 로 깨지면서 finally 절이 실행됩니다. server.py 의 /chat/stream/safe 에는 다음이 들어 있습니다.

```python
except asyncio.CancelledError:
    print("[safe] cancelled by client")
    raise
finally:
    print("[safe] generator finished or cancelled")
```

여기서 핵심은 두 가지입니다.

1. 엔드포인트와 generator 가 모두 async 여야 클라이언트 끊김이 곧장 CancelledError 로 전파됩니다. sync 로 두면 OpenAI 응답을 기다리는 스레드가 블로킹 중이라 finally 가 늦게(혹은 안) 실행됩니다.
2. CancelledError 는 잡아서 로그만 남기고 반드시 raise 해야 합니다. 삼키면 FastAPI 가 정상 응답으로 오해합니다.

**해볼 것**
1. 아래 셀을 실행. 응답이 흐르기 시작하면 일정 분량만 받고 break.
2. 서버 터미널 로그에 [safe] cancelled by client 와 [safe] generator finished or cancelled 가 차례로 찍히는지 확인.
3. 정상 종료(끝까지 받음) 시에도 [safe] generator finished or cancelled 한 줄이 찍힘. 즉 finally 는 두 경우 모두에 호출됨.

이 패턴이 중요한 이유: 실제 서비스에서는 generator 안에서 외부 리소스(파일, DB 커넥션, 로그 핸들 등) 를 잡고 있을 수 있는데, 클라이언트가 끊었을 때 그걸 풀어주지 않으면 누수가 납니다. finally 가 그 자리가 됩니다.

In [11]:
# 클라이언트가 도중에 끊는 시나리오: 처음 5 개 청크만 받고 break
import httpx

with httpx.Client(timeout=60.0) as c:
    with c.stream(
        "POST",
        f"{SERVER_BASE}/chat/stream/safe",
        json={"message": "한국의 역사를 가능한 한 길게 설명해줘"},
    ) as r:
        count = 0
        for piece in r.iter_text():
            print(piece, end="", flush=True)
            count += 1
            if count >= 5:
                print("\n[client] break")
                break
print("--- end ---")
print("이제 서버 터미널을 보세요. [safe] cancelled by client → [safe] generator finished or cancelled 순으로 찍혀 있어야 합니다.")

한국의 역사는 매우
[client] break
--- end ---
이제 서버 터미널을 보세요. [safe] cancelled by client → [safe] generator finished or cancelled 순으로 찍혀 있어야 합니다.


# TODO

지금까지 본 패턴을 직접 손에 익히기 위한 과제입니다.

## TODO 1. /chat/stream/safe 에 타임아웃 적용

`/chat/stream/safe` 의 OpenAI 호출에 명시적 `timeout` 을 걸어, 타임아웃이 나면 본문 끝에 `[ERROR] ...` 한 줄이 흐르게 합니다.

- `create(...)` 에 `timeout` 인자 추가 (짧게, 예 3.0)
- 타임아웃 예외는 기존 `except Exception` 에서 잡혀 `[ERROR] ...` 로 나감
- `CancelledError`·`finally` 로그는 그대로

아래 셀에서 `safe` 안 `timeout` 빈칸을 채우고 실행하면 server.py 가 갱신됩니다.

**검증** — 긴 답변을 요구하며 timeout 을 짧게 잡으면 본문 끝에 `[ERROR] APITimeoutError: ...`, 짧은 답변이면 `[DONE]`.

In [12]:
# TODO 1 — /chat/stream/safe 의 OpenAI 호출에 timeout 적용
safe = '''

@app.post("/chat/stream/safe")
async def chat_stream_safe(req: ChatRequest):
    async def gen():
        try:
            stream = await client.chat.completions.create(
                model=req.model or MODEL,
                messages=[{"role": "user", "content": req.message}],
                temperature=req.temperature if req.temperature is not None else 1.0,
                stream=True,
                
                # TODO 1: create 에 timeout 인자 추가 (짧게, 예 3.0)
                None,
            )
            async for chunk in stream:
                if not chunk.choices:
                    continue
                piece = getattr(chunk.choices[0].delta, "content", None)
                if not piece:
                    continue
                yield piece
            yield "\\n[DONE]"
        except asyncio.CancelledError:
            print("[safe] cancelled by client")
            raise
        except Exception as e:
            yield f"\\n[ERROR] {type(e).__name__}: {e}"
        finally:
            print("[safe] generator finished or cancelled")
    return StreamingResponse(gen(), media_type="text/plain")
'''

Path("server.py").write_text(head + safe + break_ + robust, encoding="utf-8")
print("server.py 갱신 — /chat/stream/safe 에 timeout 적용")

server.py 갱신 — /chat/stream/safe 에 timeout 적용


In [15]:
# TODO 1 호출: 긴 답변을 요구해서 타임아웃 유도
import httpx

with httpx.Client(timeout=7.0) as c:
    with c.stream(
        "POST",
        f"{SERVER_BASE}/chat/stream/safe",
        json={"message": "한국의 역사를 가능한 한 길게 설명해줘"},
    ) as r:
        print("status:", r.status_code)
        for piece in r.iter_text():
            print(piece, end="", flush=True)
print("\n--- end ---")

status: 200

[ERROR] APITimeoutError: Request timed out.
--- end ---


## TODO 2. /chat/stream/break 에 빈 응답 감지 추가

`/chat/stream/break` 는 지금 토큰을 한 번도 안 흘립니다. 받는 쪽이 "빈 응답"임을 알 수 있도록, generator 가 끝났는데 한 번도 yield 한 적이 없으면 `[EMPTY]` 한 줄을 흘리게 만듭니다.

- `produced` 플래그로 토큰을 흘렸는지 추적
- 루프가 끝난 뒤 `produced` 가 False 면 `[EMPTY]` + 줄바꿈 yield

아래 셀의 `break_` 안 `# TODO 2-1·2-2·2-3` 빈칸을 채우고 실행하세요.

**검증** — `/chat/stream/break` 호출 시 본문에 `[EMPTY]` 한 줄이 흐르면 OK (status 200).

In [16]:
# TODO 2
break_ = '''

@app.post("/chat/stream/break")
async def chat_stream_break(req: ChatRequest):
    """빈 응답만 흘리는 더미 + 빈 응답 감지 패턴."""
    async def gen():
        # TODO 2-1: 토큰을 한 번이라도 흘렸는지 추적할 플래그 (초기값)
        produced = None
        
        try:
            for _ in range(3):
                piece = ""  # 실제로는 LLM chunk 가 전부 빈 문자열인 상황
                if piece:
                
                    # TODO 2-2: 토큰을 흘렸으니 플래그 갱신
                    None

                    yield piece
            # TODO 2-3: 한 번도 안 흘렸으면 "[EMPTY]" + "\\n" 두 줄 yield
            None
        finally:
            print("[break] generator finished")
    return StreamingResponse(gen(), media_type="text/plain")
'''

Path("server.py").write_text(head + safe + break_ + robust, encoding="utf-8")
print("server.py 갱신 — /chat/stream/break 에 [EMPTY] 감지 추가")

server.py 갱신 — /chat/stream/break 에 [EMPTY] 감지 추가


In [17]:
# TODO 2 호출: 빈 응답일 때 [EMPTY] 한 줄이 흐르는지 확인
import httpx

with httpx.Client(timeout=30.0) as c:
    with c.stream(
        "POST",
        f"{SERVER_BASE}/chat/stream/break",
        json={"message": "비어있을 예정"},
    ) as r:
        print("status:", r.status_code)
        body = ""
        for piece in r.iter_text():
            body += piece
            print(piece, end="", flush=True)
print("\n--- end ---")
print("본문 길이:", len(body))
assert "[EMPTY]" in body, "빈 응답일 때 본문에 [EMPTY] 마커가 포함되어야 합니다."

status: 200
[EMPTY]

--- end ---
본문 길이: 8


## TODO 3. /chat/stream/robust 구현

지금까지의 패턴(타임아웃·빈 응답 감지·예외·정상 종료)을 한 엔드포인트에 모은 `/chat/stream/robust` 를 만듭니다.

- `async def` 엔드포인트 + async generator
- 호출에 `timeout` 적용(TODO 1), `produced` 플래그 추적(TODO 2)
- 정상 종료 → `[DONE]`, 한 번도 안 흘렸으면 → `[EMPTY]`, 예외 → `[ERROR] {타입}: {메시지}`
- `CancelledError` 는 잡아 로그 남기고 `raise`, `finally` 에 정리 로그

아래 셀의 `robust` 안 `# TODO 3-1·3-2·3-3` 빈칸을 채우세요 — `safe` 엔드포인트가 좋은 본보기입니다.

**검증** — 정상 질문 → `[DONE]` / 잘못된 model → `[ERROR] ...` / 클라이언트 break → 터미널에 `[robust] cancelled...` 로그.

In [18]:
# TODO 3 — /chat/stream/robust 신규
robust = '''

@app.post("/chat/stream/robust")
async def chat_stream_robust(req: ChatRequest):
    async def gen():
        produced = False
        try:
            # TODO 3-1: AsyncOpenAI 스트리밍 호출 — safe 처럼, 단 timeout 포함
            stream = None

            async for chunk in stream:
                if not chunk.choices:
                    continue
                piece = getattr(chunk.choices[0].delta, "content", None)
                if not piece:
                    continue
                produced = True
                yield piece

            # TODO 3-2: produced 면 "\\n[DONE]", 아니면 "[EMPTY]" + "\\n"
            None

        except asyncio.CancelledError:
            print("[robust] cancelled by client")
            raise
        except Exception as e:
            # TODO 3-3: 예외 마커 — "[ERROR] {타입}: {메시지}" + "\\n"
            None

        finally:
            print("[robust] generator finished or cancelled")
    return StreamingResponse(gen(), media_type="text/plain")
'''

Path("server.py").write_text(head + safe + break_ + robust, encoding="utf-8")
print("server.py 갱신 — /chat/stream/robust 추가")

server.py 갱신 — /chat/stream/robust 추가


In [19]:
# TODO 3 호출 1 - 정상 질문: 본문 끝이 [DONE]
import httpx

with httpx.Client(timeout=60.0) as c:
    with c.stream(
        "POST",
        f"{SERVER_BASE}/chat/stream/robust",
        json={"message": "FastAPI 한 줄 요약 부탁해"},
    ) as r:
        print("status:", r.status_code)
        body = ""
        for piece in r.iter_text():
            body += piece
            print(piece, end="", flush=True)
print("\n--- end ---")
assert body.rstrip().endswith("[DONE]"), "정상 응답은 [DONE] 으로 끝나야 합니다."

status: 200
FastAPI는 **Python으로 빠르고 쉽게 고성능 REST API를 만들 수 있게 해주는 현대적인 웹 프레임워크**입니다.
[DONE]
--- end ---


In [20]:
# TODO 3 호출 2 - 잘못된 model: 본문 끝이 [ERROR] ...
import httpx

with httpx.Client(timeout=60.0) as c:
    with c.stream(
        "POST",
        f"{SERVER_BASE}/chat/stream/robust",
        json={"message": "테스트", "model": "this-model-does-not-exist"},
    ) as r:
        print("status:", r.status_code)
        body = ""
        for piece in r.iter_text():
            body += piece
            print(piece, end="", flush=True)
print("\n--- end ---")
assert "[ERROR]" in body, "호출 실패 시 본문에 [ERROR] 마커가 있어야 합니다."

status: 200

[ERROR] BadRequestError: Error code: 400 - {'detail': "Model 'this-model-does-not-exist' is not allowed. Allowed models: ['openai/gpt-5.4']"}
--- end ---


In [21]:
# TODO 3 호출 3 - 클라이언트 중단: 서버 터미널 로그 확인
import httpx

with httpx.Client(timeout=60.0) as c:
    with c.stream(
        "POST",
        f"{SERVER_BASE}/chat/stream/robust",
        json={"message": "한국의 역사를 가능한 한 길게 설명해줘"},
    ) as r:
        count = 0
        for piece in r.iter_text():
            print(piece, end="", flush=True)
            count += 1
            if count >= 5:
                print("\n[client] break")
                break
print("--- end ---")
print("이제 서버 터미널에서 [robust] cancelled by client → [robust] generator finished or cancelled 순으로 로그를 확인하세요.")

한국의 역사는 매우
[client] break
--- end ---
이제 서버 터미널에서 [robust] cancelled by client → [robust] generator finished or cancelled 순으로 로그를 확인하세요.


## 정리

- 스트리밍 엔드포인트는 async def 엔드포인트 + async generator + AsyncOpenAI 로 통일해야 클라이언트 중단이 곧장 CancelledError 로 전파되고 finally 가 즉시 실행된다.
- CancelledError 는 잡아서 로그만 남기고 반드시 raise 한다. 삼키면 FastAPI 가 정상 응답으로 오해한다.
- 서버 호출(client.chat.completions.create) 자체에 timeout 을 걸어 두면 무한 대기를 막을 수 있고, 클라이언트 측 httpx.Client(timeout=...) 와 함께 두 겹으로 방어하는 것이 안전하다.
- generator 가 한 번도 yield 하지 못한 채 끝나면 본문 길이 0 의 정상 200 응답이 되어 빈 응답과 정상 응답을 구분할 수 없다. produced 플래그로 추적해서 [EMPTY] 한 줄을 흘려주면 본문 마커로 구분된다.
- 본문 마지막 줄을 [DONE] / [EMPTY] / [ERROR] 셋 중 하나로 고정해 두면, 받는 쪽이 어떤 사유로 끝났는지 한 줄만 보고 판단할 수 있다.